In [1]:
import pandas as pd
import re
import hashlib

In [2]:
chat = "private"

In [3]:
def hash_author(author: str) -> str:
    """Return a short anonymized identifier for an author."""
    # Use SHA1 hash and take first 8 characters
    h = hashlib.sha1(author.encode('utf-8')).hexdigest()[:8]
    return str(h)

In [4]:
df = pd.read_csv(f"../data/raw/{chat}.csv")

# anonimize
df['Author'] = df['Author'].apply(hash_author)

# remove attachments
df['Content'] = df.apply(
    lambda row: f"<ATTACH> {row['Content'] if pd.notna(row['Content']) else ''}".strip()
    if pd.notna(row['Attachments']) and row['Attachments'].strip() != '' 
    else row['Content'],
    axis=1
)

# remove links
url_regex = r"(?P<url>https?://[^\s]+)"
df['Content'] = df['Content'].apply(
	lambda x: re.sub(url_regex, "<URL>", x)
	if isinstance(x, str) else x,
)

# date_mask = df['Date'].between('2021-01-01', '2023-12-31')
date_mask = df['Date'].between('2024-01-01', '2025-12-31')
df = df.loc[date_mask]

df = df[['Author', 'Content']]
df['Content'] = df['Content'].astype("str")
print(df.head())

          Author                        Content
177681  e1688f3c                       <ATTACH>
177682  e1688f3c                    happy polla
177683  fa09cb53                       <ATTACH>
177684  fa09cb53  private firework show earlier
177685  fa09cb53               just le got home


In [ ]:
df.to_csv(f'../data/processed/{chat}_rag.csv', index=False)